# A* search

_Artificial Intelligence (CS221)_

**Uniform cost search with a hint. Aim toward the goal and reach it much faster.**

Uniform cost search explores in all directions. It does not know where the goal is.

     A* search adds a heuristic: a cheap guess of the remaining distance to the goal.

---

This notebook builds **A\* search** one piece at a time. Run each cell, read the note above it to see how the heuristic steers the search toward the goal, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python

### Step 1 — Define the grid, the goal, and the heuristic

We search a small 5×7 grid for a shortest path from `start` to `goal`, moving one cell at a time (cost 1 per step). The heuristic `h(p)` is the **Manhattan distance** to the goal — the number of grid steps you'd need if nothing were in the way.

Manhattan distance is *admissible*: it never over-estimates the true remaining cost, which is exactly the property A\* needs to be guaranteed to return an optimal path.

In [ ]:
import heapq

ROWS, COLS = 5, 7
start, goal = (2, 1), (2, 5)

def h(p):                                   # Manhattan distance: admissible on a grid
    return abs(p[0]-goal[0]) + abs(p[1]-goal[1])

### Step 2 — Generate legal neighbours and seed the frontier

`neighbors(p)` yields the up/down/left/right cells that stay inside the grid. The **frontier** `pq` is a priority queue of entries `(f, g, node, path)`, where `g` is the cost paid so far and `f = g + h` is the estimated total cost through that node. Python's `heapq` always pops the *lowest* `f` first, so A\* expands the most promising cell next.

`best_g` remembers the cheapest `g` we've found for each cell, so we never re-open a cell along a more expensive route.

In [ ]:
def neighbors(p):
    r, c = p
    for dr, dc in ((1,0),(-1,0),(0,1),(0,-1)):
        nr, nc = r+dr, c+dc
        if 0 <= nr < ROWS and 0 <= nc < COLS:
            yield (nr, nc)

pq = [(h(start), 0, start, [start])]        # (f = g+h, g, node, path)
best_g = {start: 0}
expanded = 0

### Step 3 — Expand the frontier until the goal pops

Each loop pops the lowest-`f` cell and counts it as *expanded*. If it's the goal, we're done — and because the heuristic is admissible, the first time the goal is popped its path is guaranteed optimal. Otherwise we relax each neighbour: a step costs `ng = g + 1`, and if that beats the best known cost to that neighbour we record it and push it with its own `f = ng + h(v)`.

The final `expanded` count is usually well below the 35 total cells — the heuristic kept the search aimed at the goal.

In [ ]:
while pq:
    f, g, u, path = heapq.heappop(pq)
    expanded += 1
    if u == goal:
        print("optimal path length:", len(path) - 1)
        print("path:", path)
        print("cells expanded:", expanded, "of", ROWS * COLS)
        break
    for v in neighbors(u):
        ng = g + 1                          # cost to reach v through u
        if ng < best_g.get(v, 1e9):         # only keep the cheapest route to v
            best_g[v] = ng
            heapq.heappush(pq, (ng + h(v), ng, v, path + [v]))

## Visualize the data & results

_On a downtown street grid, which blocks does A\* explore first when routing a delivery van?_

### Step 1 — Score every block by f = g + h

We reframe the grid as downtown blocks: a `depot` and a `customer`. For *every* block we compute `f = g + h`, where `g` is the Manhattan distance back to the depot (cost already paid to get there) and `h` is the Manhattan distance forward to the customer (the heuristic estimate of what's left). Lower-`f` blocks are the ones A\* would expand first, so this field is a map of the search's priorities.

In [ ]:
ROWS, COLS = 5, 7
depot, customer = (2, 1), (2, 5)        # 3rd-and-B, 3rd-and-F

def manh(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

f = np.zeros((ROWS, COLS))
for r in range(ROWS):
    for c in range(COLS):
        g_cost = manh((r, c), depot)        # cost already paid to reach this block
        h_est = manh((r, c), customer)      # heuristic estimate of cost remaining
        f[r, c] = g_cost + h_est

### Step 2 — Draw the f-field as a heatmap

We render the `f` values as a heatmap over the street grid, labelling the axes with cross-streets and printing each block's `f` value on top. The dark, low-`f` corridor running between depot and customer is exactly the band of blocks A\* prioritises — it explores along the straight line toward the goal rather than fanning out in every direction.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(f, cmap="viridis")

ax.set_yticks(range(5))
ax.set_yticklabels(["1st","2nd","3rd","4th","5th"])
ax.set_xticks(range(7))
ax.set_xticklabels(list("ABCDEFG"))

for r in range(ROWS):
    for c in range(COLS):
        ax.text(c, r, int(f[r, c]), ha="center", va="center", color="white")

ax.set_title("f = g + h over the downtown block grid")
fig.colorbar(im, ax=ax)
plt.show()